# 55. 원판을 ONNX로 바꾸면 검출 품질과 처리 시간이 어떻게 달라질까?

베이스라인 학습에 들어가지 않은 6명의 검증 사진 39장에 같은 입력을 넣습니다. 각 사진에는 원판 정답이 있습니다. 점수 0.001 이상 후보로 mAP를 구하고, 운영 기준 0.10에서 직접 검출이 달라진 프레임도 확인합니다. 시간 연결은 적용하지 않습니다. 품질은 이번에 채점하고, 시간은 46번의 영상 687프레임 측정표에서 읽습니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from wlcv.config import ROOT
from wlcv.evaluation.runtime_map import compare_map

# 다른 Mac에서 재현할 때는 46번 노트북에서 만든 그래프·측정표 폴더를 지정합니다.
reference = ROOT / "outputs/onnx_plate_parity/20261125-194508-519898"
output = ROOT / "outputs/runtime_quality" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True)
summary, thresholds, frames = compare_map(reference, output)
display(summary.round(6))
display(frames.groupby("runtime")[["targets", "candidates", "direct_tp", "direct_fp", "direct_fn"]].sum())

평균이 같아도 특정 IoU에서 차이가 날 수 있습니다. 오른쪽 그림은 원판 모델 추론만 잰 시간이며 왼쪽 그림과 표본이 다릅니다. P95는 측정한 프레임 중 95%가 해당 시간 안에 끝났다는 뜻입니다.

In [ ]:
latency = pd.read_csv(reference / "latency.csv")
times = latency.groupby("runtime").ms.quantile([0.50, 0.95]).unstack()
times.columns = ["p50_ms", "p95_ms"]
speed = times.copy()
# 모든 프레임 수를 누적 모델 실행 시간(초)으로 나눕니다.
speed["model_fps"] = 1000 / latency.groupby("runtime").ms.mean()
display(speed.round(2))
per_photo = frames.pivot(index="image", columns="runtime", values=["direct_tp", "direct_fp", "direct_fn"])
changed = per_photo.xs("pytorch", axis=1, level="runtime").ne(per_photo.xs("onnx", axis=1, level="runtime"))
print(f"직접 검출 정오표가 달라진 사진: {changed.any(axis=1).sum()}/{len(changed)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
ap = thresholds.pivot(index="iou", columns="runtime", values="ap")
axes[0].plot(ap.index, ap.pytorch, label="PyTorch", color="#156286", linewidth=3)
axes[0].plot(ap.index, ap.onnx, label="ONNX Runtime", color="#b44212", linestyle="--", marker=".")
axes[0].set(xlabel="IoU threshold", ylabel="Plate AP", ylim=(0, 1.04), title="39 annotated validation photos")
axes[0].legend()
times.loc[["pytorch", "onnx"]].plot.bar(ax=axes[1], rot=0, color=["#156286", "#a4c9d5"])
axes[1].set(xlabel="", ylabel="Inference time (ms)", title="687 video frames on Apple M2 CPU")
fig.savefig(output / "quality_latency.png", dpi=150)
plt.show()

In [ ]:
summary.to_csv(output / "map.csv", index=False)
thresholds.to_csv(output / "ap_thresholds.csv", index=False)
frames.to_csv(output / "photo_counts.csv", index=False)
speed.to_csv(output / "latency_summary.csv")
source = json.loads((reference / "settings.json").read_text())
source["reference_run"] = reference.name
source["images"] = len(per_photo)
source["dataset"] = "고정 원판 검증 JPEG 39장 (6명)"
(output / "settings.json").write_text(json.dumps(source, ensure_ascii=False, indent=2))
print(output)

후속 GPU 실험에서는 동일 GPU의 PyTorch CUDA FP32를 기준으로 ONNX Runtime CUDA FP32, TensorRT FP32·FP16을 비교합니다. 실제 입력을 사용할 수 있는 경우에만 이 검출 채점과 자세·궤적 검증, 오버레이 저장 시간을 GPU 결과와 연결할 수 있습니다.